# Lista 5: Połączenia Residualne i Transfer Learning

## Spis treści
1. **ZADANIE 1**: Implementacja modułu residualnego
2. **ZADANIE 2**: Transfer Learning z ResNet
   - Eksperyment 1: Linear Probing
   - Eksperyment 2: Partial Fine-tuning
   - Eksperyment 3: Full Fine-tuning
   - Eksperyment 4: Training from Scratch
3. Porównanie i analiza wyników

---
## ZADANIE 1: Implementacja Modułu Residualnego

Zaimplementuj blok residualny zgodnie ze schematem:
```
x -> Conv -> ReLU -> Conv -> ( + x ) -> ReLU
```

**Kroki:**
1. Policz F(x) = Conv -> ReLU -> Conv
2. Dodaj skip connection: out = F(x) + x
3. Zastosuj ReLU na końcu i zwróć wynik

In [ ]:
# Przygotowanie dummy input (NIE MODYFIKOWAĆ)
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ustalmy powtarzalność
torch.manual_seed(0)

# Dummy batch obrazów: (batch_size, channels, height, width)
B, C, H, W = 4, 16, 32, 32
x = torch.randn(B, C, H, W)
print("x shape:", x.shape)

In [ ]:
# TODO: Implementacja ResidualBlock

class ResidualBlock(nn.Module):
    """
    Blok residualny zgodny ze schematem:
        x -> Conv -> ReLU -> Conv -> ( + x ) -> ReLU
    Zakładamy, że wejście i wyjście mają te same wymiary (C, H, W).
    """
    def __init__(self, channels: int, kernel_size: int = 3):
        super().__init__()
        padding = kernel_size // 2  # żeby zachować H,W dla kernel=3

        self.conv1 = nn.Conv2d(channels, channels, kernel_size=kernel_size, padding=padding, bias=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=kernel_size, padding=padding, bias=True)
        self.relu = nn.ReLU(inplace=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 1: policz F(x) = Conv -> ReLU -> Conv
        fx = None  # Zastąp None odpowiednią implementacją
        
        # TODO 2: dodaj skip connection: out = F(x) + x
        out = None  # Zastąp None odpowiednią implementacją
        
        # TODO 3: zastosuj ReLU na końcu i zwróć wynik
        out = None  # Zastąp None odpowiednią implementacją
        
        return out

In [ ]:
# Testy ResidualBlock (NIE MODYFIKOWAĆ - uruchom po implementacji)

def zero_out_conv(conv: nn.Conv2d):
    with torch.no_grad():
        conv.weight.zero_()
        if conv.bias is not None:
            conv.bias.zero_()

# 1) Test wymiarów
block = ResidualBlock(channels=C)
y = block(x)
print("y shape:", y.shape)
assert y.shape == x.shape, f"Shape mismatch: got {y.shape}, expected {x.shape}"
print("✓ Shape test passed")

# 2) Test semantyczny: jeśli F(x)=0 -> y = ReLU(x)
zero_out_conv(block.conv1)
zero_out_conv(block.conv2)
y2 = block(x)
expected = torch.relu(x)
max_diff = (y2 - expected).abs().max().item()
print("max |y2 - ReLU(x)| =", max_diff)
assert torch.allclose(y2, expected, atol=1e-6), "Residual behavior test failed"
print("✓ Residual behavior test passed")

# 3) Test gradientu
x_req = x.clone().detach().requires_grad_(True)
y3 = block(x_req)
loss = y3.sum()
loss.backward()
grad_norm = x_req.grad.norm().item()
print("x grad norm:", grad_norm)
assert grad_norm > 0, "Gradient flow test failed"
print("✓ Gradient flow test passed")

print("\n🎉 Wszystkie testy przeszły pomyślnie!")

---
## ZADANIE 2: Transfer Learning z ResNet

### Przygotowanie środowiska i danych

In [ ]:
# Import bibliotek
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

In [ ]:
# TODO: Przygotowanie danych (CIFAR-10 lub FashionMNIST)
# 
# Kroki:
# 1. Wybierz zbiór danych (CIFAR-10 lub FashionMNIST)
# 2. Zdefiniuj transformacje (resize do 224x224, normalizacja ImageNet)
# 3. Załaduj train/val/test splits
# 4. Stwórz DataLoadery

# Normalizacja ImageNet
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])

# TODO: Zdefiniuj transformacje dla train i val/test
train_transforms = None  # Twoja implementacja

val_transforms = None  # Twoja implementacja

# TODO: Załaduj dataset
# train_dataset = ...
# val_dataset = ...
# test_dataset = ...

# TODO: Stwórz DataLoadery
# train_loader = ...
# val_loader = ...
# test_loader = ...

# num_classes = ?  # Ustaw odpowiednią liczbę klas

In [ ]:
# TODO: Funkcje pomocnicze do treningu i ewaluacji
# 
# Zaimplementuj:
# - train_epoch(model, loader, criterion, optimizer, device)
# - evaluate(model, loader, criterion, device)
# - train_model(model, train_loader, val_loader, epochs, lr, device)

def train_epoch(model, loader, criterion, optimizer, device):
    """Trening jednej epoki"""
    # TODO: Twoja implementacja
    pass

def evaluate(model, loader, criterion, device):
    """Ewaluacja modelu"""
    # TODO: Twoja implementacja
    pass

def train_model(model, train_loader, val_loader, num_epochs, learning_rate, device):
    """Pełny proces treningu z zapisem historii"""
    # TODO: Twoja implementacja
    # Zwróć historię treningową (train_losses, val_losses, train_accs, val_accs)
    pass

### Eksperyment 1: Linear Probing

**Cel:** Zamroź wszystkie warstwy poza ostatnią. Trenuj tylko klasyfikator.

In [ ]:
# TODO: Eksperyment 1 - Linear Probing
#
# Kroki:
# 1. Załaduj ResNet18/34 z wagami pretrenowanymi (weights='DEFAULT' lub 'IMAGENET1K_V1')
# 2. Zamroź wszystkie parametry (requires_grad = False)
# 3. Zastąp ostatnią warstwę fc nową warstwą z odpowiednią liczbą klas
# 4. Odmroź tylko nową warstwę fc
# 5. Wytrenuj model

print("=" * 50)
print("EKSPERYMENT 1: Linear Probing")
print("=" * 50)

# TODO: Załaduj model
# model_exp1 = models.resnet18(weights=...)

# TODO: Zamroź wszystkie parametry
# for param in model_exp1.parameters():
#     param.requires_grad = False

# TODO: Zastąp ostatnią warstwę
# num_features = model_exp1.fc.in_features
# model_exp1.fc = nn.Linear(num_features, num_classes)

# TODO: Przenieś model na device
# model_exp1 = model_exp1.to(device)

# TODO: Trenuj model (np. 10 epok, lr=0.001)
# history_exp1 = train_model(model_exp1, train_loader, val_loader, 
#                            num_epochs=10, learning_rate=0.001, device=device)

### Eksperyment 2: Partial Fine-tuning

**Cel:** Zamroź większość modelu, odmroź ostatni blok ResNet + klasyfikator.

In [ ]:
# TODO: Eksperyment 2 - Partial Fine-tuning
#
# Kroki:
# 1. Załaduj ResNet18/34 z wagami pretrenowanymi
# 2. Zamroź wszystkie parametry
# 3. Zastąp ostatnią warstwę fc
# 4. Odmroź ostatni blok layer4 (lub layer3 i layer4 dla ResNet18)
# 5. Odmroź fc
# 6. Wytrenuj model

print("=" * 50)
print("EKSPERYMENT 2: Partial Fine-tuning")
print("=" * 50)

# TODO: Załaduj model
# model_exp2 = models.resnet18(weights=...)

# TODO: Zamroź wszystkie parametry
# for param in model_exp2.parameters():
#     param.requires_grad = False

# TODO: Zastąp ostatnią warstwę
# num_features = model_exp2.fc.in_features
# model_exp2.fc = nn.Linear(num_features, num_classes)

# TODO: Odmroź ostatni blok (layer4 dla ResNet)
# for param in model_exp2.layer4.parameters():
#     param.requires_grad = True

# TODO: Odmroź fc
# for param in model_exp2.fc.parameters():
#     param.requires_grad = True

# TODO: Przenieś model na device
# model_exp2 = model_exp2.to(device)

# TODO: Trenuj model (np. 10 epok, lr=0.001)
# history_exp2 = train_model(model_exp2, train_loader, val_loader,
#                            num_epochs=10, learning_rate=0.001, device=device)

### Eksperyment 3: Full Fine-tuning

**Cel:** Użyj modelu pretrenowanego, odmroź wszystkie warstwy i dotrenuj cały model.

In [ ]:
# TODO: Eksperyment 3 - Full Fine-tuning
#
# Kroki:
# 1. Załaduj ResNet18/34 z wagami pretrenowanymi
# 2. Zastąp ostatnią warstwę fc
# 3. Wszystkie parametry powinny być odmrożone (domyślnie są)
# 4. Wytrenuj cały model (możesz użyć mniejszego lr niż przy treningu od zera)

print("=" * 50)
print("EKSPERYMENT 3: Full Fine-tuning")
print("=" * 50)

# TODO: Załaduj model
# model_exp3 = models.resnet18(weights=...)

# TODO: Zastąp ostatnią warstwę
# num_features = model_exp3.fc.in_features
# model_exp3.fc = nn.Linear(num_features, num_classes)

# TODO: Przenieś model na device
# model_exp3 = model_exp3.to(device)

# TODO: Trenuj model (np. 5-10 epok, lr=0.0001 - mniejszy niż przy treningu od zera)
# history_exp3 = train_model(model_exp3, train_loader, val_loader,
#                            num_epochs=10, learning_rate=0.0001, device=device)

### Eksperyment 4: Training from Scratch

**Cel:** Trenuj ResNet od zera bez wag pretrenowanych. Wystarczy kilka pierwszych epok.

In [ ]:
# TODO: Eksperyment 4 - Training from Scratch
#
# Kroki:
# 1. Załaduj ResNet18/34 BEZ wag pretrenowanych (weights=None)
# 2. Zastąp ostatnią warstwę fc
# 3. Wytrenuj model od zera (kilka epok wystarczy do obserwacji)

print("=" * 50)
print("EKSPERYMENT 4: Training from Scratch")
print("=" * 50)

# TODO: Załaduj model BEZ wag pretrenowanych
# model_exp4 = models.resnet18(weights=None)

# TODO: Zastąp ostatnią warstwę
# num_features = model_exp4.fc.in_features
# model_exp4.fc = nn.Linear(num_features, num_classes)

# TODO: Przenieś model na device
# model_exp4 = model_exp4.to(device)

# TODO: Trenuj model (np. 5 epok, lr=0.001)
# history_exp4 = train_model(model_exp4, train_loader, val_loader,
#                            num_epochs=5, learning_rate=0.001, device=device)

---
## Porównanie i Analiza Wyników

Po zakończeniu wszystkich eksperymentów, porównaj wyniki i wyciągnij wnioski.

In [ ]:
# TODO: Wizualizacja porównawcza
#
# Stwórz wykresy porównujące:
# - Train loss dla wszystkich eksperymentów
# - Validation loss dla wszystkich eksperymentów
# - Train accuracy dla wszystkich eksperymentów
# - Validation accuracy dla wszystkich eksperymentów

# Przykładowa struktura:
# fig, axes = plt.subplots(2, 2, figsize=(15, 10))
# 
# # Train Loss
# axes[0, 0].plot(history_exp1['train_loss'], label='Exp1: Linear Probing')
# axes[0, 0].plot(history_exp2['train_loss'], label='Exp2: Partial Fine-tuning')
# axes[0, 0].plot(history_exp3['train_loss'], label='Exp3: Full Fine-tuning')
# axes[0, 0].plot(history_exp4['train_loss'], label='Exp4: From Scratch')
# axes[0, 0].set_title('Training Loss')
# axes[0, 0].set_xlabel('Epoch')
# axes[0, 0].set_ylabel('Loss')
# axes[0, 0].legend()
# axes[0, 0].grid(True)
# 
# # ... podobnie dla pozostałych metryk
# 
# plt.tight_layout()
# plt.show()

In [ ]:
# TODO: Ewaluacja na zbiorze testowym
#
# Oceń wszystkie modele na zbiorze testowym i porównaj wyniki końcowe

# Przykład:
# test_acc_exp1 = evaluate_accuracy(model_exp1, test_loader, device)
# test_acc_exp2 = evaluate_accuracy(model_exp2, test_loader, device)
# test_acc_exp3 = evaluate_accuracy(model_exp3, test_loader, device)
# test_acc_exp4 = evaluate_accuracy(model_exp4, test_loader, device)
#
# print("\n" + "="*50)
# print("WYNIKI NA ZBIORZE TESTOWYM")
# print("="*50)
# print(f"Eksperyment 1 (Linear Probing):      {test_acc_exp1:.2f}%")
# print(f"Eksperyment 2 (Partial Fine-tuning): {test_acc_exp2:.2f}%")
# print(f"Eksperyment 3 (Full Fine-tuning):    {test_acc_exp3:.2f}%")
# print(f"Eksperyment 4 (From Scratch):        {test_acc_exp4:.2f}%")

### Analiza i Wnioski

**Pytania do rozważenia:**

1. **Który eksperyment osiągnął najlepsze wyniki na zbiorze walidacyjnym/testowym?**

2. **Jak wyglądają krzywe uczenia (train loss vs val loss) dla każdego eksperymentu?**
   - Czy zauważasz overfitting w którymś z przypadków?
   - Który eksperyment stabilizuje się najszybciej?

3. **Porównanie Linear Probing vs Partial Fine-tuning:**
   - Jak duża jest różnica w wynikach?
   - Czy odmrożenie jednego bloku daje znaczącą poprawę?

4. **Porównanie Full Fine-tuning vs Training from Scratch:**
   - Jak bardzo pretrenowanie wpływa na końcowy wynik?
   - Czy model od zera "dogania" model pretrenowany przy dłuższym treningu?

5. **Szybkość zbieżności:**
   - Który eksperyment osiąga dobre wyniki najszybciej?
   - Jak pretrenowanie wpływa na dynamikę uczenia?

6. **Praktyczne wnioski:**
   - W jakich sytuacjach zastosowałbyś każde z podejść?
   - Jakie są trade-offy między różnymi strategiami?

### Twoje wnioski

TODO: Zapisz swoje obserwacje i wnioski z eksperymentów:

1. **Najlepsze wyniki:**
   - ...

2. **Analiza krzywych uczenia:**
   - ...

3. **Porównanie podejść:**
   - ...

4. **Wpływ pretrenowania:**
   - ...

5. **Praktyczne zastosowania:**
   - ...